# Проект 1. Влияние совместного использования подписки на удержание клиентов и выручку «Прайма»

## Бизнес-вопрос
Влияет ли использование дополнительных мест в подписке на удержание клиентов, их активность и выручку «Прайма»?
## Итоговая гипотеза 
Гипотеза: Клиенты, которые активнее используют доступные места на своей подписке, имеют более высокое удержание и активность, а также обеспечивают большую выручку «Прайма», чем клиенты, которые используют подписку преимущественно самостоятельно
## Доп вопрос для декомпозиции
За счёт каких факторов меняется выручка «Прайма»: количества активных подписок, доли платящих клиентов, частоты платежей или среднего чека?

### 1. Подключение

In [ ]:
# как в дз
import json
import sys

import numpy as np
import pandas as pd
from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Моменты времени считаем в UTC: так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

In [ ]:
import sys
!{sys.executable} -m ensurepip --upgrade
!{sys.executable} -m pip install seaborn
!{sys.executable} -m pip install plotly

## 2. Анализ доступных данных в БД для выбора гипотезы

In [ ]:
# Список таблиц, доступных в базе данных

tables = pd.read_sql("""
    SELECT
        table_schema,
        table_name
    FROM information_schema.tables
    WHERE table_type = 'BASE TABLE'
      AND table_schema NOT IN ('pg_catalog', 'information_schema')
    ORDER BY table_schema, table_name
""", engine)

display(tables)

In [ ]:
# Количество строк в основных таблицах
table_names = [
    "clients",
    "subscriptions",
    "tariffs",
    "subscription_members",
    "payments",
    "service_usage",
    "services",
    "transactions",
    "cashback",
    "partners",
    "support_tickets",
    "marketing_spend",
]

table_sizes = []

for table_name in table_names:
    count = pd.read_sql(
        f"SELECT COUNT(*) AS n FROM prime.{table_name}",
        engine
    ).iloc[0]["n"]

    table_sizes.append({
        "table_name": table_name,
        "row_count": count
    })

table_sizes = pd.DataFrame(table_sizes)
display(table_sizes)

In [ ]:
# Временные границы основных событий

date_ranges = {
    "clients": """
        SELECT
            MIN(registered_at) AS min_date,
            MAX(registered_at) AS max_date
        FROM prime.clients
    """,
    "subscriptions": """
        SELECT
            MIN(started_at) AS min_date,
            MAX(started_at) AS max_date
        FROM prime.subscriptions
    """,
    "payments": """
        SELECT
            MIN(paid_at) AS min_date,
            MAX(paid_at) AS max_date
        FROM prime.payments
    """,
    "service_usage": """
        SELECT
            MIN(started_at) AS min_date,
            MAX(started_at) AS max_date
        FROM prime.service_usage
    """,
    "transactions": """
        SELECT
            MIN(occurred_at) AS min_date,
            MAX(occurred_at) AS max_date
        FROM prime.transactions
    """,
    "support_tickets": """
        SELECT
            MIN(created_at) AS min_date,
            MAX(created_at) AS max_date
        FROM prime.support_tickets
    """,
}

date_summary = []

for table_name, sql in date_ranges.items():
    result = pd.read_sql(sql, engine).iloc[0]

    date_summary.append({
        "table_name": table_name,
        "min_date": result["min_date"],
        "max_date": result["max_date"]
    })

date_summary = pd.DataFrame(date_summary)
display(date_summary)

In [ ]:
tariffs = pd.read_sql("""
    SELECT
        tariff_id,
        tariff_name,
        monthly_fee,
        cashback_rate,
        share_slots,
        valid_from,
        valid_to
    FROM prime.tariffs
    ORDER BY tariff_id, valid_from
""", engine)

display(tariffs)

In [ ]:
tariff_overlaps = pd.read_sql("""
    SELECT
        t1.tariff_name,
        t1.valid_from AS period_1_start,
        t1.valid_to AS period_1_end,
        t1.monthly_fee AS price_1,
        t2.valid_from AS period_2_start,
        t2.valid_to AS period_2_end,
        t2.monthly_fee AS price_2
    FROM prime.tariffs t1
    JOIN prime.tariffs t2
      ON t1.tariff_name = t2.tariff_name
     AND t1.valid_from < t2.valid_from
     AND t1.valid_to >= t2.valid_from
    ORDER BY t1.tariff_name, t1.valid_from
""", engine)

display(tariff_overlaps)

In [ ]:
subscriptions_by_tariff = pd.read_sql("""
    SELECT
        s.tariff_id,
        t.tariff_name,
        t.share_slots,
        COUNT(*) AS subscription_count,
        COUNT(DISTINCT s.owner_client_id) AS owner_count,
        MIN(s.started_at) AS first_subscription,
        MAX(s.started_at) AS last_subscription
    FROM prime.subscriptions s
    LEFT JOIN prime.tariffs t
      ON s.tariff_id = t.tariff_id
    GROUP BY
        s.tariff_id,
        t.tariff_name,
        t.share_slots
    ORDER BY s.tariff_id, t.share_slots
""", engine)

display(subscriptions_by_tariff)

### 3. Бизнес-вопрос №1. Связан ли переход клиента с тарифа с меньшим количеством мест на тариф с большим количеством мест с повышением удержания клиентов, их активности и выручки «Прайма»?

In [ ]:
with engine.connect() as conn:
    subscription_history = pd.read_sql("""
        SELECT
            owner_client_id,
            COUNT(*) AS subscription_count,
            COUNT(DISTINCT tariff_id) AS tariff_versions,
            MIN(started_at) AS first_subscription,
            MAX(started_at) AS last_subscription
        FROM prime.subscriptions
        GROUP BY owner_client_id
    """, conn)
display(subscription_history.head())

display(
    subscription_history["subscription_count"]
    .value_counts()
    .sort_index()
    .rename_axis("subscription_count")
    .reset_index(name="owner_count")
)

print(
    "Клиентов с несколькими подписками:",
    (subscription_history["subscription_count"] > 1).sum()
)

In [ ]:
# История тарифов 
tariffs = q("""
    SELECT *
    FROM prime.tariffs
    ORDER BY tariff_id, valid_from
""")

ref = tariffs.copy()

# Приводим даты справочника к UTC,
# чтобы сравнивать их с временными метками платежей
for col in ("valid_from", "valid_to"):
    ref[col] = pd.to_datetime(ref[col]).dt.tz_localize("UTC")

# 1. Ищем пересекающиеся интервалы одного тарифа
a = ref.reset_index(drop=True).reset_index(names="row_id")

pairs = a.merge(
    a,
    on="tariff_id",
    suffixes=("_a", "_b")
)

overlap = pairs[
    (pairs["row_id_a"] < pairs["row_id_b"])
    & (pairs["valid_from_a"] <= pairs["valid_to_b"])
    & (pairs["valid_from_b"] <= pairs["valid_to_a"])
]

display(
    overlap[
        [
            "row_id_a", "tariff_id",
            "valid_from_a", "valid_to_a", "monthly_fee_a",
            "row_id_b", "valid_from_b", "valid_to_b", "monthly_fee_b"
        ]
    ]
)

# 2. Находим запись, которая чаще всего участвует в пересечениях
bad_id = pd.concat([
    overlap["row_id_a"],
    overlap["row_id_b"]
]).value_counts().idxmax()

print("Индекс подозрительной записи:", bad_id)

# 3. Создаём исправленную копию справочника
ref_fixed = (
    a[a["row_id"] != bad_id]
    .drop(columns="row_id")
    .copy()
)

display(ref_fixed.sort_values(["tariff_id", "valid_from"]))

In [ ]:
q("""
    SELECT
        COUNT(*) AS subscription_rows,
        COUNT(DISTINCT owner_client_id) AS unique_owners,
        COUNT(*) - COUNT(DISTINCT owner_client_id) AS extra_subscriptions
    FROM prime.subscriptions
""")

## 4. Промежуточные результаты исследования

### Гипотеза №1: клиенты на тарифе с большим количеством мест имеют более высокое удержание, активность и выручку, особенно если используют доступные места совместно с другими пользователями.


В базе 500 000 клиентов и 333 905 подписок. Представлены два тарифных уровня: «Базовый» на 3 места и «Расширенный» на 5 мест. Я столкнулась с проблемой с историей тарифов (решила ее как в семинаре), а также с ограничением в данных.
В таблице subscriptions 333 905 записей и столько же уникальных владельцев. Поэтому проследить последовательный переход одного владельца с базового тарифа на расширенный по этой таблице невозможно.

В связи с этим мною было принято решение скорректировать гипотезу


## 5. Формироание новой гипотезы и начало ее проверки

### Гипотеза № 2

Клиенты, которые активнее используют доступные места на своей подписке, имеют более высокое удержание и активность, а также обеспечивают большую выручку «Прайма», чем клиенты, которые используют подписку преимущественно самостоятельно.


### Зерно для гипотезы №2
Одна строка в таблице для каждой изметрики

- subscriptions — одна строка на подписку 
- subscription_members — одна строка на участие клиента в подписке
- tariffs — исторический справочник тарифов
- payments — одна строка на платеж 
- service_usage — одна строка на событие использования сервиса

На первом этапе основное зерно — одна строка на подписку (subscription_id). На этом уровне мы определим количество участников и предварительно оценим заполненность доступных мест.
Для дальнейшего сравнения активности и выручки данные будут агрегироваться до уровня подписка × календарный месяц. Это позволит сопоставлять показатели за одинаковые периоды и не допускать дублирования платежей и событий использования при объединении таблиц.
При расчёте заполненности необходимо учитывать даты присоединения и выхода участников (joined_at, left_at). Текущее количество участников нельзя автоматически использовать как показатель заполненности в прошлые месяцы.

Вывод о подтверждении или опровержении гипотезы будет сделан после расчёта метрик удержания, активности и выручки.

In [ ]:
subscriptions = q("""
    SELECT
        subscription_id,
        owner_client_id,
        tariff_id,
        started_at,
        ended_at,
        is_trial,
        cancel_reason
    FROM prime.subscriptions
""")

In [ ]:
# Участники подписок
members = q("""
    SELECT
        subscription_id,
        client_id,
        joined_at,
        left_at
    FROM prime.subscription_members
""")

In [ ]:
# 1. Количество участников по каждой подписке
# Владелец не включён в subscription_members

member_stats = (
    members
    .groupby("subscription_id")
    .agg(members_ever=("client_id", "nunique"))
    .join(
        members[members["left_at"].isna()]
        .groupby("subscription_id")
        .agg(members_current=("client_id", "nunique"))).fillna(0).reset_index())

# Справочник тарифов
tariff_dim = (
    ref_fixed
    .groupby("tariff_id", as_index=False)
    .agg(
        tariff_name=("tariff_name", "max"),
        share_slots=("share_slots", "max")))


In [ ]:
# Объединяем таблицы
sharing = (
    subscriptions
    .merge(tariff_dim, on="tariff_id", how="left", validate="m:1")
    .merge(member_stats, on="subscription_id", how="left", validate="1:1"))

# Заполняем отсутствующих участников нулями
sharing[["members_ever", "members_current"]] = (
    sharing[["members_ever", "members_current"]]
    .fillna(0)
    .astype(int)
)

sharing["users_current"] = 1 + sharing["members_current"]

sharing["occupancy_rate_current"] = (
    sharing["users_current"]
    / sharing["share_slots"].replace(0, np.nan))


In [ ]:
print("Количество строк:", len(sharing))
print("Уникальных подписок:", sharing["subscription_id"].nunique())
print(
    "Количество строк совпадает с числом уникальных подписок:",
    len(sharing) == sharing["subscription_id"].nunique()
)

sharing.head()

In [ ]:
# Группы подписок по числу текущих пользователей
sharing["sharing_group"] = pd.cut(
    sharing["users_current"],
    bins=[0, 1, 2, 3, 5, float("inf")],
    labels=[
        "1 пользователь",
        "2 пользователя",
        "3 пользователя",
        "4–5 пользователей",
        "6+ пользователей"
    ],
    include_lowest=True
)

# Распределение подписок по тарифам и группам
sharing_distribution = (
    sharing
    .groupby(
        ["tariff_name", "sharing_group"],
        observed=False,
        dropna=False
    )
    .agg(
        subscriptions=("subscription_id", "nunique"),
        owners=("owner_client_id", "nunique")
    )
    .reset_index()
)

display(sharing_distribution)

In [ ]:
# Заполненность подписок по тарифам

# Доля подписок, у которых есть хотя бы один текущий участник
sharing["has_current_members"] = sharing["members_current"].gt(0)

occupancy_summary = (
    sharing
    .groupby("tariff_name", dropna=False)
    .agg(
        subscriptions=("subscription_id", "nunique"),
        avg_current_users=("users_current", "mean"),
        median_current_users=("users_current", "median"),
        avg_occupancy=("occupancy_rate_current", "mean"),
        share_with_members=("has_current_members", "mean")).reset_index())

# Переводим доли в проценты
occupancy_summary["avg_occupancy_pct"] = (occupancy_summary["avg_occupancy"] * 100).round(1)

occupancy_summary["share_with_members_pct"] = (occupancy_summary["share_with_members"] * 100).round(1)

occupancy_summary

- Базовый тариф: 207 578 подписок, в среднем 2,23 пользователя на подписку, средняя текущая занятость мест — 74,4%
- Расширенный тариф: 126 327 подписок, в среднем 2,85 пользователя, средняя текущая занятость — 57,0%
- Доля подписок хотя бы с одним участником помимо владельца — около 66,8% на базовом, 68,3% на расширенном 
- На расширенном тарифе заметно больше подписок с 4–5 пользователями — 36 629, и есть 9 377 подписок с 6+ пользователями

In [ ]:
# Проверка 1. Подписки, где число пользователей превышает лимит тарифа

over_limit = sharing[sharing["share_slots"].notna() & (sharing["users_current"] > sharing["share_slots"])].copy()

print("Подписок с превышением лимита:", len(over_limit))
print( "Доля таких подписок, %:",
    round(len(over_limit) / len(sharing) * 100, 2))

display(
    over_limit[
        ["subscription_id", "tariff_name","share_slots","members_current","users_current"]].head(10))


# Проверка 2. Минимум и максимум числа пользователей по тарифам

users_check = (
    sharing.groupby("tariff_name", dropna=False)
    .agg(
        subscriptions=("subscription_id", "nunique"),
        min_users=("users_current", "min"),
        max_users=("users_current", "max")).reset_index())

display(users_check)

In [ ]:
# Проверяем интервалы участия пользователей в подписках

members_check = q("""
    SELECT
        COUNT(*) AS membership_rows,
        COUNT(DISTINCT subscription_id) AS subscriptions_with_members,
        COUNT(DISTINCT client_id) AS unique_clients,
        COUNT(*) FILTER (WHERE joined_at IS NULL) AS missing_joined_at,
        COUNT(*) FILTER (WHERE left_at IS NULL) AS current_members,
        COUNT(*) FILTER (
            WHERE left_at IS NOT NULL
              AND left_at < joined_at
        ) AS invalid_intervals
    FROM prime.subscription_members
""")

display(members_check)

- В таблице sharing — 333 905 строк и столько же уникальных подписок. Зерно сохранилось
- У 42 249 подписок (12,65%) число пользователей превышает лимит тарифа
- На базовом тарифе максимум — 4 пользователя при лимите 3; на расширенном — 6 при лимите 5
- В subscription_members — 555 858 записей участия и 233 733 подписки с участниками

In [ ]:
# Проверяем повторные записи одного пользователя в рамках подписки
membership_duplicates = q("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (subscription_id, client_id)) AS unique_subscription_clients,
        COUNT(*) - COUNT(DISTINCT (subscription_id, client_id))
            AS extra_rows
    FROM prime.subscription_members
""")

display(membership_duplicates)

In [ ]:
#Проверка пересечения периодов участия
overlapping_memberships = q("""
    SELECT
        a.subscription_id,
        a.client_id,
        a.joined_at AS joined_at_1,
        a.left_at AS left_at_1,
        b.joined_at AS joined_at_2,
        b.left_at AS left_at_2
    FROM prime.subscription_members a
    JOIN prime.subscription_members b
      ON a.subscription_id = b.subscription_id
     AND a.client_id = b.client_id
     AND a.joined_at < b.joined_at
     AND (a.left_at IS NULL OR a.left_at > b.joined_at)
    LIMIT 20
""")

print("Найдено примеров пересекающихся периодов:", len(overlapping_memberships))
display(overlapping_memberships)

- Агрегация и LEFT JOIN member_stats к subscriptions не размножили строки
- Превышение лимита мест нельзя объяснить повторными записями одного участника в одной подписке

In [ ]:
# Проверяем текущих участников у завершённых подписок
ended_with_current_members = q("""
    SELECT
        COUNT(DISTINCT s.subscription_id) AS ended_subscriptions,
        COUNT(DISTINCT s.subscription_id)
            FILTER (WHERE m.client_id IS NOT NULL) AS ended_with_current_members
    FROM prime.subscriptions s
    LEFT JOIN prime.subscription_members m
        ON s.subscription_id = m.subscription_id
       AND m.left_at IS NULL
    WHERE s.ended_at IS NOT NULL
""")

display(ended_with_current_members)

In [ ]:
# Проверяем статус подписок и даты завершения участия

subscription_status_check = q("""
    SELECT
        CASE
            WHEN s.ended_at IS NULL THEN 'Подписка не завершена'
            ELSE 'Подписка завершена'
        END AS subscription_status,

        COUNT(DISTINCT s.subscription_id) AS subscriptions,

        COUNT(DISTINCT s.subscription_id)
            FILTER (WHERE m.client_id IS NOT NULL) AS with_open_memberships,

        COUNT(DISTINCT s.subscription_id)
            FILTER (
                WHERE m.client_id IS NOT NULL
                  AND m.joined_at > s.ended_at
            ) AS member_joined_after_subscription_end

    FROM prime.subscriptions s
    LEFT JOIN prime.subscription_members m
        ON s.subscription_id = m.subscription_id
       AND m.left_at IS NULL
    GROUP BY 1
    ORDER BY 1
""")

display(subscription_status_check)

In [ ]:
# Пересчитываем текущих участников только для незавершённых подписок

sharing_current = q("""
    WITH member_stats AS (
        SELECT
            s.subscription_id,
            COUNT(DISTINCT m.client_id)
                FILTER (WHERE m.left_at IS NULL) AS members_current
        FROM prime.subscriptions s
        LEFT JOIN prime.subscription_members m
            ON s.subscription_id = m.subscription_id
        WHERE s.ended_at IS NULL
        GROUP BY s.subscription_id
    ),
    tariff_dim AS (
        SELECT
            tariff_id,
            MAX(tariff_name) AS tariff_name,
            MAX(share_slots) AS share_slots
        FROM prime.tariffs
        GROUP BY tariff_id
    )
    SELECT
        s.subscription_id,
        s.owner_client_id,
        t.tariff_name,
        t.share_slots,
        COALESCE(m.members_current, 0) AS members_current,
        1 + COALESCE(m.members_current, 0) AS users_current,
        (1 + COALESCE(m.members_current, 0))::numeric
            / NULLIF(t.share_slots, 0) AS occupancy_rate_current
    FROM prime.subscriptions s
    LEFT JOIN tariff_dim t
        ON s.tariff_id = t.tariff_id
    LEFT JOIN member_stats m
        ON s.subscription_id = m.subscription_id
    WHERE s.ended_at IS NULL
""")

print("Строк:", len(sharing_current))
print(
    "Уникальных подписок:",
    sharing_current["subscription_id"].nunique()
)

display(
    sharing_current.groupby("tariff_name", dropna=False)
    .agg(
        subscriptions=("subscription_id", "nunique"),
        avg_users=("users_current", "mean"),
        avg_occupancy=("occupancy_rate_current", "mean"),
        max_users=("users_current", "max")
    )
    .reset_index()
)


- В subscriptions 333 905 строк и 333 905 уникальных подписок — дублирования нет
- После объединения с предварительно агрегированными участниками и тарифами число строк сохранилось. В этом запросе размножения подписок нет
- В subscription_members нет повторов пар «подписка — участник» и не обнаружено пересекающихся периодов участия
- Подтвердили, что владельца нужно учитывать отдельно, а платежи нельзя напрямую соединять с участниками — это может задублировать выручку
- Обнаружили особенности данных: у части завершённых подписок остались участники с left_at IS NULL, а у 7 739 подписок есть участники, добавленные после завершения подписки

share_slots — это количество мест для участников, с которыми поделились подпиской, поэтому владельца не прибавляем при расчёте занятости мест. Владельца учитываем отдельно, когда считаем общее число пользователей подписки.

In [ ]:
# Текущая занятость мест по незавершённым подпискам

# Оставляем только незавершённые подписки
sharing_current = sharing[sharing["ended_at"].isna()].copy()

# Занятость мест
sharing_current["occupancy_rate_current"] = (sharing_current["members_current"]/ sharing_current["share_slots"].replace(0, np.nan))

print("Строк:", len(sharing_current))
print(
    "Уникальных подписок:",
    sharing_current["subscription_id"].nunique())

# Проверяем превышение лимита дополнительных участников
over_limit = sharing_current[sharing_current["share_slots"].notna()& (sharing_current["members_current"] > sharing_current["share_slots"])]

print("Подписок с превышением лимита:", len(over_limit))
print(
    "Доля таких подписок, %:",
    round(len(over_limit) / len(sharing_current) * 100, 2))

# Сводка по тарифам
occupancy_summary = (sharing_current.groupby("tariff_name", dropna=False).agg(
        subscriptions=("subscription_id", "nunique"),
        avg_members=("members_current", "mean"),
        avg_users=("users_current", "mean"),
        avg_occupancy=("occupancy_rate_current", "mean")
    ).reset_index())

occupancy_summary["avg_occupancy_pct"] = (occupancy_summary["avg_occupancy"] * 100).round(1)

display(occupancy_summary)

**Микровывод: оценка занятости мест в подписке**

У расширенного тарифа в среднем больше участников, но доля занятых мест немного ниже

Для оценки занятости мест мы объединили подписки с агрегированной статистикой по участникам и справочником тарифов. Агрегация участников до уровня подписки позволила избежать дублирования записей. В расчёт вошли 167 978 незавершённых подписок.
- Базовый тариф: в среднем 1,23 участника помимо владельца при 3 доступных местах для участников. Средняя занятость доступных мест — 41,1%.
- Расширенный тариф: в среднем 1,85 участника при 5 доступных местах. Средняя занятость — 37,0%.
- Превышений лимита участников по текущим данным не обнаружено.
Таким образом, на расширенном тарифе в среднем больше участников, но доля занятых мест ниже. При этом left_at IS NULL отражает состояние записи об участнике, а не подтверждённое использование сервиса.

**Таблица usage является очень тяжелой, поэтому для нее оставим подзапросы в sql**

In [ ]:
activity_check = q("""
    SELECT
        s.subscription_id,
        m.client_id,
        COUNT(su.usage_id) AS usage_events,
        COUNT(DISTINCT DATE_TRUNC('month', su.started_at)) AS active_months
    FROM prime.subscriptions s
    JOIN prime.subscription_members m
        ON m.subscription_id = s.subscription_id
    LEFT JOIN prime.service_usage su
        ON su.client_id = m.client_id
        AND su.started_at >= GREATEST(
            m.joined_at, s.started_at,
            CURRENT_TIMESTAMP - INTERVAL '90 days'
        )
        AND su.started_at <= CURRENT_TIMESTAMP
    WHERE s.ended_at IS NULL
        AND m.left_at IS NULL
        AND m.joined_at <= CURRENT_TIMESTAMP
        AND m.joined_at >= s.started_at
    GROUP BY s.subscription_id, m.client_id
""")

In [ ]:
# Проверка фактической активности участников подписки

activity_check["has_usage"] = activity_check["usage_events"] > 0

activity_summary = pd.DataFrame([{
    "participants_in_records": len(activity_check),
    "participants_with_usage": activity_check["has_usage"].sum(),
    "participants_without_usage": (~activity_check["has_usage"]).sum(),
    "avg_usage_events": activity_check["usage_events"].mean(),
    "avg_active_months": activity_check["active_months"].mean(),
    "pct_with_usage": (
        activity_check["has_usage"].mean() * 100
    )}]).round(2)

display(activity_summary)

За последние 90 дней у 65% участников, числящихся в текущих незавершённых подписках, зарегистрировано хотя бы одно событие использования сервиса. У 35% участников таких событий не обнаружено. Это подтверждает, что наличие записи об участии в подписке не равнозначно фактическому использованию.

Это показывает, что наличие участника в подписке не обязательно означает фактическое использование сервиса. Для проверки гипотезы далее необходимо сравнить удержание, активность и выручку между подписками с разным количеством фактически используемых мест, учитывая тариф.

## 6. Сравнение подписок с разным уровнем совместного использования

**Cколько подписок продолжает действовать спустя определённое время после оформления**

- 30-й день — проверяем, осталась ли подписка активной через месяц
- 60-й день — проверяем, осталась ли подписка активной через два месяца
- 90-й день — проверяем, осталась ли подписка активной через три месяца

In [ ]:
# Оставляем подписки, которым уже исполнилось 90 дней
subscriptions_h2 = subscriptions.copy()
subscriptions_h2["started_at"] = pd.to_datetime(subscriptions_h2["started_at"], utc=True)
subscriptions_h2["ended_at"] = pd.to_datetime(subscriptions_h2["ended_at"], utc=True)
subscriptions_h2 = subscriptions_h2[(subscriptions_h2["started_at"] >= "2025-01-01") &(subscriptions_h2["started_at"] + pd.Timedelta(days=90) <= pd.Timestamp.now(tz="UTC"))].copy()

# Добавляем данные о тарифах
tariff_info = (ref_fixed[["tariff_id", "tariff_name", "share_slots"]].drop_duplicates())

subscriptions_h2 = subscriptions_h2.merge(
    tariff_info,on="tariff_id", how="left",validate="many_to_one")

# Даты проверки удержания
for day in [30, 60, 90]:
    subscriptions_h2[f"day_{day}"] = (
        subscriptions_h2["started_at"] + pd.Timedelta(days=day)
    )

# Число дополнительных участников на 30-й день
members_day_30 = members.merge(
    subscriptions_h2[["subscription_id", "day_30"]],
    on="subscription_id",
    how="inner")
# Оставляем только участников, которые присоединились к подписке не позже 30-го дня и всё ещё состояли в ней на 30-й день после оформления
members_day_30 = members_day_30[(members_day_30["joined_at"] <= members_day_30["day_30"]) & (members_day_30["left_at"].isna()
        | (members_day_30["left_at"] > members_day_30["day_30"]))]

members_day_30 = (members_day_30.groupby("subscription_id")["client_id"].nunique().reset_index(name="members_day_30"))

In [ ]:
owner_activity_h2 = q("""
    SELECT
        s.subscription_id,
        COUNT(su.usage_id) AS owner_usage_events,
        COUNT(DISTINCT DATE_TRUNC('month', su.started_at))
            AS owner_active_months
    FROM prime.subscriptions s
    LEFT JOIN prime.service_usage su
        ON su.client_id = s.owner_client_id
       AND su.started_at >= s.started_at
       AND su.started_at < s.started_at + INTERVAL '90 days'
    WHERE s.started_at >= TIMESTAMP '2025-01-01'
      AND s.started_at + INTERVAL '90 days' <= CURRENT_TIMESTAMP
    GROUP BY s.subscription_id
""")

In [ ]:
payments_h2 = q("""
    SELECT
        p.subscription_id,
        p.amount,
        p.status,
        p.paid_at
    FROM prime.payments p
    JOIN prime.subscriptions s
        ON s.subscription_id = p.subscription_id
    WHERE s.started_at >= TIMESTAMP '2025-01-01'
        AND s.started_at + INTERVAL '90 days' <= CURRENT_TIMESTAMP
        AND p.paid_at >= s.started_at
        AND p.paid_at < s.started_at + INTERVAL '90 days'
""")

In [ ]:
# Считаем выручку и успешные платежи
revenue_90d = (payments_h2[payments_h2["status"] == "success"].groupby("subscription_id").agg(revenue_90d=("amount", "sum"),
        successful_payments_90d=("amount", "count")).reset_index())

In [ ]:
#Объединение данных и группы участников

hypothesis_2_detail = (
    subscriptions_h2
    .merge(members_day_30, on="subscription_id", how="left")
    .merge(owner_activity_h2, on="subscription_id", how="left")
    .merge(revenue_90d, on="subscription_id", how="left"))

cols = ["members_day_30", "owner_usage_events","owner_active_months", "revenue_90d", "successful_payments_90d"]
# Пропуски заменяем нулями
hypothesis_2_detail[cols] = hypothesis_2_detail[cols].fillna(0)

# Группируем подписки по числу участников 
hypothesis_2_detail["sharing_group"] = pd.cut(hypothesis_2_detail["members_day_30"],
    bins=[-1, 0, 1, float("inf")],
    labels=["0 участников", "1 участник", "2+ участника"])

# Удержание на 30-й, 60-й и 90-й день
for day in [30, 60, 90]:
    hypothesis_2_detail[f"retained_day_{day}"] = (
        hypothesis_2_detail["ended_at"].isna()
        | (hypothesis_2_detail["ended_at"] >= hypothesis_2_detail[f"day_{day}"])).astype(int)

# Активен ли владелец
hypothesis_2_detail["owner_active"] = (hypothesis_2_detail["owner_usage_events"] > 0).astype(int)


In [ ]:
# Сводная таблица
hypothesis_2 = hypothesis_2_detail.groupby(
    ["tariff_name", "sharing_group"]
).agg(
    subscriptions=("subscription_id", "nunique"),
    retention_30d_pct=("retained_day_30", "mean"),
    retention_60d_pct=("retained_day_60", "mean"),
    retention_90d_pct=("retained_day_90", "mean"),
    owners_active_pct=("owner_active", "mean"),
    avg_owner_usage_events=("owner_usage_events", "mean"),
    avg_revenue_90d=("revenue_90d", "mean"),
    avg_payments_90d=("successful_payments_90d", "mean")
).reset_index()

hypothesis_2[
    ["retention_30d_pct", "retention_60d_pct",
     "retention_90d_pct", "owners_active_pct"]
] *= 100

display(hypothesis_2.round(2))

Для каждой подписки определили:
- количество дополнительных участников, состоявших в подписке на 30-й день
- удержание подписки на 30-й, 60-й и 90-й день
- активность владельца за первые 90 дней: количество событий использования сервиса и число активных месяцев
- выручку и количество успешных платежей за первые 90 дней

Я тут проверяю, приносит ли совместное использование подписки дополнительную ценность бизнесу «Прайм». Для этого сравниваю подписки, которыми владелец пользуется самостоятельно, с подписками, доступ к которым он разделяет с другими участниками. Сравнение проводится отдельно внутри каждого тарифа, чтобы определить, связано ли совместное использование с более высокими показателями удержания, активности и выручки.

In [ ]:
display(
    hypothesis_2_detail.groupby(
        ["tariff_name", "sharing_group"]
    )["subscription_id"].nunique().reset_index()
)

## График 1. Удержание клиентов в зависимости от типа подписки

In [ ]:
retention_plot = hypothesis_2.melt(id_vars=["tariff_name", "sharing_group"],
    value_vars=[
        "retention_30d_pct",
        "retention_60d_pct",
        "retention_90d_pct"],
    var_name="period",
    value_name="retention_pct")

retention_plot["period"] = retention_plot["period"].map({
    "retention_30d_pct": "30 дней",
    "retention_60d_pct": "60 дней",
    "retention_90d_pct": "90 дней"
})

In [ ]:
import plotly.express as px

fig = px.line(
    retention_plot,
    x="period",
    y="retention_pct",
    color="tariff_name",
    line_dash="sharing_group",
    markers=True,
    title="Удержание подписок по тарифам и количеству участников",
    labels={
        "period": "Срок после оформления",
        "retention_pct": "Удержание, %",
        "tariff_name": "Тариф",
        "sharing_group": "Количество участников"
    },
    category_orders={"period": ["30 дней", "60 дней", "90 дней"]},
    template="plotly_white"
)

fig.update_layout(yaxis=dict(range=[0, 100], ticksuffix="%"))
fig.show()

**Вывод**: гипотеза о том, что активное использование дополнительных мест повышает удержание, активность владельцев и выручку «Прайма», не получила явного подтверждения.
- Удержание. На 90-й день оно составляет примерно 90,4–90,5% во всех группах. 
- Активность владельцев. Доля активных владельцев близка во всех группах — около 82,5–83,4%. 
- Выручка. В тарифе «Расширенный» средняя выручка за 90 дней примерно вдвое выше, чем в «Базовом». Однако внутри каждого тарифа различия между группами по количеству участников минимальны.

Итог: количество дополнительных участников само по себе не демонстрирует заметной связи с удержанием, активностью владельцев или средней выручкой. 

## 7. Помесячный анализ 

In [ ]:
sharing_groups = hypothesis_2_detail[["subscription_id", "sharing_group"]].drop_duplicates("subscription_id")

subscriptions_h2 = subscriptions_h2.merge(sharing_groups, on="subscription_id", how="left")
#subscriptions_h2.head()

In [ ]:
# Приводим даты к одному формату
subscriptions_h2["started_at"] = pd.to_datetime(subscriptions_h2["started_at"], utc=True)
subscriptions_h2["ended_at"] = pd.to_datetime(subscriptions_h2["ended_at"], utc=True)

# Создаём таблицу месяцев
months_df= pd.DataFrame({"month":pd.date_range(
    start="2025-01-01",
    end=pd.Timestamp.now(tz="UTC").tz_localize(None),
    freq="MS"
).tz_localize("UTC")})

# Сочетаем подписки с месяцами
active_metrics = subscriptions_h2.merge(months_df, how="cross")

# Оставляем подписки, которые были активны хотя бы один день в соответствующем месяце
active_metrics = active_metrics[
    (active_metrics["started_at"] < active_metrics["month"] + pd.offsets.MonthBegin(1))
    & ( active_metrics["ended_at"].isna() | (active_metrics["ended_at"] >= active_metrics["month"]))]

# Считаем активные подписки по месяцу, тарифу и группе участников
active_metrics = active_metrics.groupby(["month", "tariff_name", "sharing_group"])["subscription_id"].nunique().reset_index(name="active_subscriptions")

In [ ]:
# Только успешные платежи с группировкой по меясца 
payments = payments_h2[payments_h2["status"] == "success"].copy()
payments["paid_at"] = pd.to_datetime(payments["paid_at"], utc=True)
payments["month"] = payments["paid_at"].dt.tz_localize(None).dt.to_period("M")

# Добавляем тариф и группу участников
payments = payments.merge(
    subscriptions_h2[["subscription_id", "tariff_name", "sharing_group"]]
    .drop_duplicates("subscription_id"),
    on="subscription_id",
    how="left")

# Считаем платежи и выручку по месяцу, тарифу и группе
payment_metrics = payments.groupby(
    ["month", "tariff_name", "sharing_group"]).agg(
    successful_payments=("amount", "size"),
    revenue=("amount", "sum")).reset_index()

payment_metrics.head()

In [ ]:
# Приводим месяц к одному формату
active_metrics["month"] = active_metrics["month"].dt.tz_localize(None).dt.to_period("M")

# Объединяем по месяцу, тарифу и группе участников
monthly_revenue = active_metrics.merge(
    payment_metrics,
    on=["month", "tariff_name", "sharing_group"],
    how="left")

# Если платежей не было, ставим 0
monthly_revenue[["successful_payments", "revenue"]] = ( monthly_revenue[["successful_payments", "revenue"]].fillna(0))

# Выручка на одну активную подписку
monthly_revenue["revenue_per_active_subscription"] = (monthly_revenue["revenue"]/ monthly_revenue["active_subscriptions"].replace(0, pd.NA))

# Преобразуем месяц в строку для отображения в Plotly
monthly_revenue["month"] = monthly_revenue["month"].astype(str)
monthly_revenue.head()

In [ ]:
# Находим месяц последнего успешного платежа,  исключая последний месяц
last_month = pd.to_datetime(payments_h2.loc[payments_h2["status"] == "success", "paid_at"], utc=True).max().tz_localize(None).to_period("M")
monthly_revenue_clean = monthly_revenue[pd.PeriodIndex(monthly_revenue["month"], freq="M") < last_month].copy()

## График 2. Месячная выручка по тарифам и группам участников

In [ ]:
monthly_revenue_clean["month"] = pd.PeriodIndex(monthly_revenue_clean["month"], freq="M").to_timestamp()

fig = px.line(
    monthly_revenue_clean,
    x="month",
    y="revenue",
    color="tariff_name",
    line_dash="sharing_group",
    markers=True,
    title="Месячная выручка по тарифам и группам участников",
    labels={"month": "Месяц", "revenue": "Выручка, ₽",
            "tariff_name": "Тариф", "sharing_group": "Группа участников"},
    template="plotly_white")

fig.update_layout(hovermode="x unified")
fig.show()

- «Расширенный» приносит больше выручки, чем «Базовый» почти на всем периоде
- В обоих тарифах больше всего выручки приносят подписки без дополнительных участников
- Выручка в целом стабильна, с заметным ростом тарифоы в марте 2026
- Большее количество участников не связано с ростом общей выручки

## График 3. Динамика активных подписок по месяцам

In [ ]:
active_metrics["month"] = pd.PeriodIndex(active_metrics["month"], freq="M").to_timestamp()

fig = px.line(
    active_metrics,
    x="month",
    y="active_subscriptions",
    color="tariff_name",
    line_dash="sharing_group",
    markers=True,
    title="Динамика активных подписок по месяцам",
    labels={"month": "Месяц", "active_subscriptions": "Активные подписки",
            "tariff_name": "Тариф", "sharing_group": "Группа участников"},
    template="plotly_white")

fig.update_layout(hovermode="x unified", xaxis_tickangle=-45)
fig.show()

- Количество активных подписок увеличивается почти весь период 
- Тариф «Базовый» с 0 дополнительных участников имеет больше всего активных подписок
- В обоих тарифах группы с 1 и 2+ участниками заметно меньше по числу активных подписок
- Все тарифы подверглись спаду в августе–сентябре 2026

## График 4. Динамика удержания подписок

In [ ]:
# Подготавливаем даты и месяц оформления подписки
data = subscriptions_h2.copy()
data["started_at"] = pd.to_datetime(data["started_at"], utc=True)
data["ended_at"] = pd.to_datetime(data["ended_at"], utc=True)
data["cohort_month"] = data["started_at"].dt.tz_localize(None).dt.to_period("M")

# Считаем размер каждой когорты
cohort_sizes = data.groupby(["cohort_month", "tariff_name", "sharing_group"])["subscription_id"].nunique()

# Проверяем удержание по месяцам жизни подписки
retention = data.merge(pd.DataFrame({"month_num": range(13)}), how="cross")
retention["check_month"] = retention["cohort_month"] + retention["month_num"]
check_date = retention["check_month"].dt.to_timestamp().dt.tz_localize("UTC")

retention = retention[
    (retention["check_month"] <= pd.Timestamp.now(tz="UTC").tz_localize(None).to_period("M"))
    & (retention["started_at"] < check_date + pd.offsets.MonthBegin(1))
    & (retention["ended_at"].isna() | (retention["ended_at"] >= check_date))]

# Считаем удержание по когортам и группам участников
retention_monthly = retention.groupby(["cohort_month", "tariff_name", "sharing_group"] + ["month_num"])["subscription_id"].nunique().reset_index(name="retained")

retention_monthly["cohort_size"] = retention_monthly.set_index(["cohort_month", "tariff_name", "sharing_group"]).index.map(cohort_sizes)

retention_monthly["retention_pct"] = (retention_monthly["retained"] / retention_monthly["cohort_size"] * 100)

# Объединяем группы участников внутри каждого тарифа
retention_plot = retention_monthly.groupby(
    ["month_num", "tariff_name"], as_index=False).agg(
    retained=("retained", "sum"),
    cohort_size=("cohort_size", "sum"))

retention_plot["retention_pct"] = (retention_plot["retained"] / retention_plot["cohort_size"] * 100)

# Строим график удержания по тарифам
fig = px.line(
    retention_plot, x="month_num", y="retention_pct",
    color="tariff_name", markers=True,
    title="Удержание подписок: Базовый и Расширенный",
    labels={
        "month_num": "Месяц жизни подписки",
        "retention_pct": "Удержание, %",
        "tariff_name": "Тариф"
    },
    template="plotly_white"
)

fig.update_layout(
    xaxis=dict(dtick=1),
    yaxis=dict(range=[0, 100]),
    hovermode="x unified"
)
fig.show()

## 8. Факторное разложение выручки по месяцам и тарифам

- active_subscriptions — сколько подписок активно в месяце
- paying_subscriptions — сколько из них совершили успешный платёж в этом месяце
- paying_share — доля платящих среди активных подписок
- payments_per_payer — среднее количество платежей на одного платящего клиента
- avg_payment — средняя сумма платежа
- revenue — общая выручка

In [ ]:
# Приводим месяц к одному формату
monthly_revenue_clean["month"] = pd.PeriodIndex(monthly_revenue_clean["month"], freq="M")

# Считаем платящие подписки по месяцам и тарифам
payments_success = payments_h2[payments_h2["status"] == "success"].copy()
payments_success["paid_at"] = pd.to_datetime(payments_success["paid_at"], utc=True, errors="coerce")
payments_success = payments_success.dropna(subset=["paid_at"])
payments_success["month"] = payments_success["paid_at"].dt.tz_localize(None).dt.to_period("M")

payments_success = payments_success.merge(
    subscriptions_h2[["subscription_id", "tariff_name"]].drop_duplicates("subscription_id"),
    on="subscription_id", how="inner")

paying_metrics = payments_success.groupby(["month", "tariff_name"])["subscription_id"].nunique().reset_index(name="paying_subscriptions")

# Объединяем данные и считаем факторы выручки
factor_data = monthly_revenue_clean.merge(paying_metrics, on=["month", "tariff_name"], how="left")
factor_data["paying_subscriptions"] = factor_data["paying_subscriptions"].fillna(0)
factor_data["paying_share"] = factor_data["paying_subscriptions"] / factor_data["active_subscriptions"].replace(0, np.nan)
factor_data["payments_per_payer"] = factor_data["successful_payments"] / factor_data["paying_subscriptions"].replace(0, np.nan)
factor_data["avg_payment"] = factor_data["revenue"] / factor_data["successful_payments"].replace(0, np.nan)

In [ ]:
# Факторы выручки
factors = ["active_subscriptions", "paying_share", "payments_per_payer", "avg_payment"]
names = ["Активные подписки", "Доля платящих подписок", "Частота платежей", "Средняя сумма платежа"]

rows = []

# Считаем вклад каждого фактора в изменение выручки по тарифам
for tariff, g in factor_data.groupby("tariff_name"):
    g = g.sort_values("month").reset_index(drop=True)

    for i in range(1, len(g)):
        prev, curr = g.iloc[i-1], g.iloc[i]
        old, new = prev[factors].to_numpy(float), curr[factors].to_numpy(float)

        if not (np.isfinite(old).all() and np.isfinite(new).all()):
            continue

        values, effects = old.copy(), []
        for j in range(len(factors)):
            before = np.prod(values)
            values[j] = new[j]
            effects.append(np.prod(values) - before)

        rows.append({
            "month": curr["month"],
            "tariff_name": tariff,
            "Изменение выручки": curr["revenue"] - prev["revenue"],
            **dict(zip(names, effects))
        })

decomposition = pd.DataFrame(rows)
display(decomposition.head().round(2))

In [ ]:
colors = {
    "Активные подписки": "purple",
    "Доля платящих подписок": "pink",
    "Частота платежей": "lightblue",
    "Средняя сумма платежа": "lightgreen",
}

## График 5. Декомпозиция изменения месячной выручки 

In [ ]:
# Считаем вклад факторов по месяцам
effects_monthly = decomposition.melt(
    id_vars=["month", "tariff_name"],
    value_vars=names,
    var_name="factor",
    value_name="effect"
).groupby(["month", "factor"], as_index=False)["effect"].sum()

# Строим график
fig = px.bar(
    effects_monthly, x=effects_monthly["month"].astype(str),
    y="effect", color="factor", barmode="relative",
    title="Факторы изменения месячной выручки",
    labels={"x": "Месяц", "effect": "Вклад в выручку, ₽", "factor": "Фактор"},
    color_discrete_map=colors, template="plotly_white"
)

fig.update_layout(xaxis_tickangle=-45)
fig.add_hline(y=0, line_color="gray")
fig.show()

In [ ]:
# Подготавливаем данные для графика
effects_long = decomposition.melt(
    id_vars=["month", "tariff_name"],
    value_vars=names,
    var_name="factor",
    value_name="effect"
).groupby(
    ["month", "tariff_name", "factor"], as_index=False
)["effect"].sum()

# Строим график декомпозиции по тарифам
fig = px.bar(
    effects_long, x=effects_long["month"].astype(str),
    y="effect", color="factor", facet_col="tariff_name",
    barmode="relative", color_discrete_map=colors,
    title="Факторы изменения выручки по тарифам",
    labels={"x": "Месяц", "effect": "Вклад в выручку, ₽", "factor": "Фактор"},
    template="plotly_white"
)

fig.update_layout(xaxis_tickangle=-45)
fig.add_hline(y=0, line_color="gray")
fig.show()

## Мем

In [ ]:
from IPython.display import Image, display

display(Image(url="https://cybersport.metaratings.ru/_images/insecure/w-1360:h-1040/aHR0cHM6Ly9zdG9yYWdlLnlhbmRleGNsb3VkLmt6L3MzLWs4cy1tZXRhcmF0aW5ncy1zdG9yYWdlLWJhY2t1cC1rei9pbWFnZXMvMmMvZmMvMmNmY2I1YWVmMjY2MDEwZjIyOTlhMjBmY2Q0MDhlZjAuanBn.webp"))

## Факторы изменения выручки
- Основной положительный вклад в изменение выручки вносит количество активных подписок
- Доля платящих подписок в большинстве месяцев оказывает отрицательное влияние
- Рост средней суммы платежа в отдельные месяцы компенсирует снижение других факторов
- В целом динамика выручки зависит прежде всего от количества активных подписок и среднего чека

# Итоги анализа бизнес-вопроса

**Итог по гипотезе**: гипотеза не получила явного подтверждения. Клиенты, которые используют дополнительные места в подписке, не демонстрируют заметно более высокого удержания, активности владельцев или средней выручки по сравнению с клиентами, использующими подписку самостоятельно. Различия между группами минимальны, поэтому считать совместное использование фактором роста выручки на основании текущих данных нельзя

**Ответ на бизнес-вопрос:**
Анализ не выявил заметных различий в удержании подписок в зависимости от количества дополнительных участников. На 90-й день удержание составляет около 90,4–90,5% во всех группах. Доля активных владельцев и количество событий использования также практически не различаются. Таким образом, первая гипотеза о том, что совместное использование повышает удержание и активность клиентов, не получила явного подтверждения

**Ответ на бизнес-вопрос:**
Помесячный анализ показал, что тариф «Расширенный» приносит больше выручки, чем «Базовый». При этом наибольшая общая выручка приходится на подписки без дополнительных участников, а наименьшая — на группу с 2+ участниками. Однако это может объясняться разным количеством подписок в группах


**Итог**: заметного преимущества совместного использования по удержанию, активности владельцев и средней выручке за первые 90 дней не обнаружено. При этом тариф «Расширенный» обеспечивает существенно большую среднюю выручку, чем «Базовый».

#### Рекомендации

1. Продолжить отслеживать удержание по группам участников. Это поможет выявить возможные различия на более длительном периоде.
2. Сравнивать выручку на одну активную подписку. Так можно корректнее оценить экономическую эффективность совместного использования.
3. Анализировать тарифы отдельно. Средняя выручка за первые 90 дней у «Расширенного» тарифа составляет около 1 074 ₽, у «Базового» — около 555 ₽.
4. Не считать совместное использование доказанным фактором роста. Текущие результаты не показывают заметного преимущества групп с дополнительными участниками, поэтому нужны дополнительные проверки перед принятием бизнес-решений.